# Fine-tuning NLLB-200 — Modele Multilingue Kirundi

Ce notebook fine-tune le modele **NLLB-200-distilled-600M** de Meta pour creer
**un seul modele** capable de traduire dans **4 directions** :

| Direction | Exemple |
|-----------|---------|
| **Kirundi → Francais** | "Amahoro" → "Paix" |
| **Francais → Kirundi** | "Comment vas-tu ?" → "Umeze gute?" |
| **Kirundi → Anglais** | "Ndagukunda" → "I love you" |
| **Anglais → Kirundi** | "Burundi is beautiful" → "Uburundi ni ciza" |

### Qu'est-ce que NLLB-200 ?
NLLB (No Language Left Behind) est un modele de traduction pre-entraine par Meta/Facebook sur **200 langues**,
dont le **Kirundi** (`run_Latn`). Le modele "distilled-600M" est une version compacte (600M de parametres)
qui tourne sur un GPU gratuit Google Colab.

### Pourquoi un seul modele multilingue ?
- **4x plus de donnees** : chaque phrase gold genere 4 paires d'entrainement au lieu d'une
- **Transfer learning** : le Francais et l'Anglais se renforcent mutuellement
- **1 seul modele** a deployer et maintenir (pas 3 modeles separes)
- Le tag de langue (`run_Latn`, `fra_Latn`, `eng_Latn`) dit au modele dans quelle direction traduire

### Pre-requis
- **Google Colab avec GPU** (Runtime > Change runtime type > T4 GPU)
- Le dataset `metadata.csv` est charge automatiquement depuis HuggingFace

### Ce qu'on obtient a la fin
Un modele **`Ijwi-ry-Ikirundi-AI/nllb-kirundi-multi`** publie sur HuggingFace,
capable de traduire dans les 4 directions ci-dessus.

> **Note** : la traduction FR↔EN n'est pas incluse — ca existe deja partout (Google Translate, DeepL, etc.).
> On se concentre sur ce qui **n'existe pas encore** : la traduction Kirundi de qualite.

---

## 1. Setup & Dependencies

**Cette cellule installe les librairies et prepare l'environnement.**

| Librairie | Role |
|-----------|------|
| `transformers` | Modeles de traduction (NLLB-200), tokenizer, Trainer |
| `datasets` | Format Dataset HuggingFace pour le training |
| `sentencepiece` | Tokenizer sous-mot utilise par NLLB |
| `sacrebleu` | Calcul du score BLEU (qualite de traduction) |
| `accelerate` | Optimisation GPU/multi-GPU |
| `huggingface_hub` | Push du modele vers HuggingFace |

Le code detecte automatiquement le GPU :
- `cuda` = GPU NVIDIA (Colab T4) — **recommande**
- `mps` = GPU Apple Silicon (Mac M1/M2) — ok mais plus lent
- `cpu` = pas de GPU — **tres lent, a eviter**

In [5]:
import sys
print(sys.executable)
print(sys.version)
import torch
print(torch.cuda.is_available(), torch.backends.mps.is_available())
#print(next(model.parameters()).device)

/Users/samandari/Documents/Personal/Projects/Kirundi_Dataset/.venv/bin/python
3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 17.0.0 (clang-1700.6.3.2)]
False True


In [6]:
!pip install -q transformers datasets sentencepiece sacrebleu accelerate huggingface_hub evaluate

import os
import pandas as pd
import numpy as np
from datetime import datetime
from datasets import Dataset, DatasetDict
from huggingface_hub import hf_hub_download, login
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import evaluate
import torch

if torch.cuda.is_available():
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
if DEVICE == 'cpu':
    print('WARNING: Pas de GPU detecte. Le training sera tres lent.')
    print('         Sur Colab: Runtime > Change runtime type > T4 GPU')
elif DEVICE == 'mps':
    print('GPU Apple Silicon detecte (MPS). OK pour le training.')

HF_DATASET_REPO = 'Ijwi-ry-Ikirundi-AI/Kirundi_Open_Speech_Dataset'
HF_MODEL_REPO = 'Ijwi-ry-Ikirundi-AI/nllb-kirundi-multi'
OUTPUT_DIR = './models'

FILE_PATH = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename='metadata.csv',
    repo_type='dataset',
)

print(f'Dataset telecharge: {FILE_PATH}')
print(f'Modeles sauvegardes dans: {OUTPUT_DIR}')

Device: mps
PyTorch: 2.11.0
GPU Apple Silicon detecte (MPS). OK pour le training.
Dataset telecharge: /Users/samandari/.cache/huggingface/hub/datasets--Ijwi-ry-Ikirundi-AI--Kirundi_Open_Speech_Dataset/snapshots/fc68c3cb08cf2021b5c50ba2dc7787d234bc79d5/metadata.csv
Modeles sauvegardes dans: ./models


## 2. Load & Prepare Data

**Charge `metadata.csv` depuis HuggingFace et genere les paires multilingues.**

**Etape 1** : On extrait les lignes **"gold"** (Kirundi + Francais + Anglais toutes remplies).

**Etape 2** : On split les phrases gold en 80/10/10 **avant** de generer les paires.
C'est important : les phrases de test ne doivent jamais apparaitre dans le train, meme dans une autre direction.

**Etape 3** : Pour chaque phrase gold, on genere **4 paires** de traduction :
| Paire | Source | Target |
|-------|--------|--------|
| 1 | Kirundi | Francais |
| 2 | Francais | Kirundi |
| 3 | Kirundi | Anglais |
| 4 | Anglais | Kirundi |

Avec ~2,911 phrases gold : **~9,312 paires train** | ~1,164 paires val | ~1,168 paires test

> Le `random_state=42` garantit que le split est reproductible : meme resultat a chaque execution.

In [8]:
df = pd.read_csv(FILE_PATH)
print(f'Total rows in metadata.csv: {len(df)}')

filled = lambda s: s.notna() & (s.astype(str).str.strip() != '')
gold = df[
    filled(df['Kirundi_Transcription'])
    & filled(df['French_Translation'])
    & filled(df['English_Translation'])
].copy()
gold = gold[['Kirundi_Transcription', 'French_Translation', 'English_Translation', 'Domain']]
gold = gold.reset_index(drop=True)

print(f'Gold phrases (KR+FR+EN): {len(gold)}')
print(f'Domains: {gold["Domain"].nunique()}')

train_gold = gold.sample(frac=0.8, random_state=42)
remaining = gold.drop(train_gold.index)
val_gold = remaining.sample(frac=0.5, random_state=42)
test_gold = remaining.drop(val_gold.index)

DIRECTIONS = [
    ('run_Latn', 'fra_Latn', 'Kirundi_Transcription', 'French_Translation'),
    ('fra_Latn', 'run_Latn', 'French_Translation', 'Kirundi_Transcription'),
    ('run_Latn', 'eng_Latn', 'Kirundi_Transcription', 'English_Translation'),
    ('eng_Latn', 'run_Latn', 'English_Translation', 'Kirundi_Transcription'),
]


def expand_directions(dataframe):
    rows = []
    for src_lang, tgt_lang, src_col, tgt_col in DIRECTIONS:
        for _, row in dataframe.iterrows():
            rows.append({
                'source': row[src_col],
                'target': row[tgt_col],
                'src_lang': src_lang,
                'tgt_lang': tgt_lang,
            })
    return pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)


train_pairs = expand_directions(train_gold)
val_pairs = expand_directions(val_gold)
test_pairs = expand_directions(test_gold)

print(f'\n--- Paires multilingues ---')
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')
print(f'\nRepartition train:')
for src, tgt, _, _ in DIRECTIONS:
    n = len(train_pairs[(train_pairs['src_lang'] == src) & (train_pairs['tgt_lang'] == tgt)])
    print(f'  {src} -> {tgt}: {n} paires')

Total rows in metadata.csv: 4737
Gold phrases (KR+FR+EN): 2911
Domains: 7

--- Paires multilingues ---
Train: 9316 | Val: 1164 | Test: 1164

Repartition train:
  run_Latn -> fra_Latn: 2329 paires
  fra_Latn -> run_Latn: 2329 paires
  run_Latn -> eng_Latn: 2329 paires
  eng_Latn -> run_Latn: 2329 paires


## 3. Hyperparametres

**Configuration du modele et du training.** Pas besoin de choisir une direction — le modele apprend les 4 simultanement.

**Hyperparametres expliques :**
- `MAX_LENGTH = 96` — un peu plus court que 128 = **plus rapide** sur Mac MPS (tu peux remettre 128 sur Colab pour la qualite max)
- `BATCH_SIZE = 1` avec `GRAD_ACCUM_STEPS = 16` — batch effectif 16
- `EPOCHS = 5` par defaut — **plus court** sur Mac ; sur Colab tu peux monter a 10
- `LEARNING_RATE = 5e-5` — vitesse d'apprentissage. Valeur standard pour le fine-tuning de modeles pre-entraines

> **Mac MPS** : l'eval avec generation (BLEU) est tres lente — le notebook fait une eval **tous les ~1200 steps** au lieu de **chaque epoch** pour gagner du temps. Sur **CUDA (Colab)** on garde une eval **par epoch**.

> **Colab T4** : ~1h-1h30 pour 5 epochs. **Mac MPS** : toujours beaucoup plus lent que le GPU NVIDIA.

In [9]:
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
MAX_LENGTH = 96
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
EPOCHS = 5
LEARNING_RATE = 5e-5

EVAL_STEPS = 1200
SAVE_STEPS = 1200

RUN_NAME = f'nllb-kirundi-multi-{datetime.now().strftime("%Y%m%d")}'

print(f'Run name:  {RUN_NAME}')
print(f'Model:     {MODEL_NAME}')
print(f'Directions: 4 (KR<->FR, KR<->EN)')
print(f'Epochs:    {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LEARNING_RATE}  |  MAX_LENGTH: {MAX_LENGTH}')
if DEVICE == 'mps':
    print(f'MPS: eval + save tous les {EVAL_STEPS} steps (pas chaque epoch) pour accelerer')
print(f'Train:     {len(train_pairs)} paires  |  Val: {len(val_pairs)}  |  Test: {len(test_pairs)}')

Run name:  nllb-kirundi-multi-20260419
Model:     facebook/nllb-200-distilled-600M
Directions: 4 (KR<->FR, KR<->EN)
Epochs:    5  |  Batch: 1  |  LR: 5e-05  |  MAX_LENGTH: 96
MPS: eval + save tous les 1200 steps (pas chaque epoch) pour accelerer
Train:     9316 paires  |  Val: 1164  |  Test: 1164


## 4. Build HuggingFace Datasets

**Convertit les DataFrames en objets `Dataset` HuggingFace.**

Chaque exemple contient 4 champs :
- `source` — la phrase a traduire
- `target` — la traduction attendue
- `src_lang` — le code langue source (ex: `run_Latn`)
- `tgt_lang` — le code langue cible (ex: `fra_Latn`)

Les champs `src_lang` et `tgt_lang` sont utilises pendant la tokenization pour dire au modele
dans quelle direction traduire chaque exemple.

In [6]:
def df_to_dataset(dataframe):
    return Dataset.from_dict({
        'source': dataframe['source'].tolist(),
        'target': dataframe['target'].tolist(),
        'src_lang': dataframe['src_lang'].tolist(),
        'tgt_lang': dataframe['tgt_lang'].tolist(),
    })

dataset = DatasetDict({
    'train': df_to_dataset(train_pairs),
    'validation': df_to_dataset(val_pairs),
    'test': df_to_dataset(test_pairs),
})

print(dataset)
print()
for i in range(min(4, len(dataset['train']))):
    ex = dataset['train'][i]
    print(f'  [{ex["src_lang"]} -> {ex["tgt_lang"]}]  {ex["source"][:50]}  ->  {ex["target"][:50]}')

DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'src_lang', 'tgt_lang'],
        num_rows: 9316
    })
    validation: Dataset({
        features: ['source', 'target', 'src_lang', 'tgt_lang'],
        num_rows: 1164
    })
    test: Dataset({
        features: ['source', 'target', 'src_lang', 'tgt_lang'],
        num_rows: 1164
    })
})

  [run_Latn -> fra_Latn]  Inda ndēnde ihūmira indyá.  ->  La cupidité engendre des ennuis.
  [eng_Latn -> run_Latn]  Excessive eating can make hunger your constant com  ->  Urya cāne ukīroga inzara.
  [eng_Latn -> run_Latn]  Do you have money to buy it?  ->  Uri n’amafaranga yo kukigura?
  [run_Latn -> fra_Latn]  Uduháganyuzo duhera níngoga, nza nûmva umukózi amb  ->  Les cure-dents se terminent très rapidement. Ensui


## 5. Load Model & Tokenizer

**Telecharge le modele NLLB-200 (~2.4 GB). Prend quelques minutes la 1ere fois.**

Le modele a 2 composants :
- **Tokenizer** : decoupe le texte en sous-mots (ex: "Amahoro" → `[Ama, horo]`).
  Chaque sous-mot a un ID numerique. Le tokenizer connait 256,000+ tokens dans 200 langues.
- **Model** : reseau de neurones Seq2Seq (encodeur-decodeur) avec 600M de parametres.
  L'encodeur comprend la phrase source, le decodeur genere la traduction mot par mot.

On verifie que le Kirundi (`run_Latn`) est bien dans le vocabulaire — si le token ID est > 0, c'est bon.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
    )
    model = model.to('cuda')
    model.gradient_checkpointing_enable()
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model = model.to(DEVICE)

print(f'Model parameters: {model.num_parameters() / 1e6:.0f}M')
print(f'Vocabulary size: {tokenizer.vocab_size}')

LANG_CODES = {'run_Latn', 'fra_Latn', 'eng_Latn'}
print('\nLanguage support:')
for lang in sorted(LANG_CODES):
    tid = tokenizer.convert_tokens_to_ids(lang)
    status = 'OK' if tid > 0 else 'MISSING'
    print(f'  {lang}: token_id={tid} [{status}]')

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Model parameters: 615M
Vocabulary size: 256204

Language support:
  eng_Latn: token_id=256047 [OK]
  fra_Latn: token_id=256057 [OK]
  run_Latn: token_id=256146 [OK]


## 6. Tokenize Dataset

**Transforme les phrases en token IDs — en respectant la langue de chaque exemple.**

Comme chaque exemple a sa propre direction (KR→FR, FR→KR, etc.), on ne peut pas tokenizer
tout le batch avec le meme `src_lang`. On tokenize **un exemple a la fois** en changeant
le `tokenizer.src_lang` pour chaque paire.

Pour chaque phrase :
1. On set `tokenizer.src_lang` au code langue source de cet exemple
2. Le tokenizer decoupe la phrase source en sous-mots + ajoute le tag langue
3. On set `tokenizer.src_lang` au code langue cible
4. On tokenize la traduction attendue (label)
5. Les tokens de padding dans les labels sont remplaces par `-100` (ignores par la loss)

> C'est un peu plus lent que le batched processing (quelques minutes), mais necessaire
> pour que chaque paire ait le bon tag de langue.

In [8]:
def preprocess_function(examples):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for src, tgt, sl, tl in zip(
        examples['source'], examples['target'],
        examples['src_lang'], examples['tgt_lang'],
    ):
        tokenizer.src_lang = sl
        enc = tokenizer(src, max_length=MAX_LENGTH, truncation=True, padding='max_length')

        tokenizer.src_lang = tl
        dec = tokenizer(tgt, max_length=MAX_LENGTH, truncation=True, padding='max_length')

        label = [
            (tok if tok != tokenizer.pad_token_id else -100)
            for tok in dec['input_ids']
        ]

        input_ids_list.append(enc['input_ids'])
        attention_mask_list.append(enc['attention_mask'])
        labels_list.append(label)

    return {
        'input_ids': input_ids_list,
        'attention_mask': attention_mask_list,
        'labels': labels_list,
    }


tokenized = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=256,
    remove_columns=['source', 'target', 'src_lang', 'tgt_lang'],
    desc='Tokenizing',
)

print(f'Tokenized train: {len(tokenized["train"])} examples')
print(f'Tokenized val:   {len(tokenized["validation"])} examples')
print(f'Tokenized test:  {len(tokenized["test"])} examples')
print(f'Features: {list(tokenized["train"].features.keys())}')

Tokenizing:   0%|          | 0/9316 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1164 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1164 [00:00<?, ? examples/s]

Tokenized train: 9316 examples
Tokenized val:   1164 examples
Tokenized test:  1164 examples
Features: ['input_ids', 'attention_mask', 'labels']


## 7. Evaluation Metric (BLEU)

**BLEU** (Bilingual Evaluation Understudy) est LE score standard pour evaluer la qualite de traduction.

Il compare la traduction du modele avec la traduction de reference (humaine) en comptant
les n-grams (sequences de mots) en commun.

| Score BLEU | Qualite |
|-----------|---------|
| < 10 | Quasi inutilisable |
| 10-20 | On comprend le sens general |
| 20-30 | Traduction acceptable |
| 30-40 | Bonne traduction |
| > 40 | Excellente traduction |

> Avec ~11,600 paires d'entrainement (4 directions), on vise un BLEU de **15-30**.
> Le score est calcule sur toutes les directions melangees. Plus on ajoute de donnees, plus il montera.

In [9]:
bleu_metric = evaluate.load('sacrebleu')


def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {'bleu': round(result['score'], 2)}


print('BLEU metric loaded.')

BLEU metric loaded.


## 8. Train

**C'est le coeur du notebook — l'entrainement du modele.**

Duree estimee :
- **Colab GPU (T4)** : ~1h-1h30 pour 5 epochs (selon charge)
- **Mac MPS** : beaucoup plus long ; l'eval BLEU (generation) a ete **espacée** (tous les ~1200 steps sur MPS)
- **CPU** : a eviter

**Ce qui se passe pendant le training :**
1. A chaque **epoch**, le modele parcourt les ~9,300 paires (par batch)
2. Sur **CUDA**, on evalue sur la validation **a chaque epoch**. Sur **MPS**, **tous les ~1200 steps** pour gagner du temps
3. Si le BLEU s'ameliore, on sauvegarde un **checkpoint** (snapshot du modele)
4. A la fin des 10 epochs, on charge automatiquement le **meilleur** checkpoint

**Parametres cles :**
- `weight_decay=0.01` — regularisation pour eviter l'overfitting
- `warmup_ratio=0.1` — les 10% premiers steps, le learning rate monte doucement (evite les explosions)
- `fp16=True` — calculs en demi-precision sur GPU = 2x plus rapide, 2x moins de memoire
- `load_best_model_at_end=True` — garantit qu'on garde le meilleur modele, pas le dernier

> **Logs** : toutes les 50 steps, le training affiche la loss (erreur). Elle doit **descendre** au fil du temps.

In [10]:
model_output_dir = os.path.join(OUTPUT_DIR, RUN_NAME)
os.makedirs(model_output_dir, exist_ok=True)

_train_args = dict(
    output_dir=model_output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=100,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    bf16=torch.cuda.is_available(),
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model='bleu',
    greater_is_better=True,
    logging_steps=200,
    save_total_limit=3,
    report_to='none',
    run_name=RUN_NAME,
)

if DEVICE == 'mps':
    _train_args.update(
        eval_strategy='steps',
        eval_steps=EVAL_STEPS,
        save_strategy='steps',
        save_steps=SAVE_STEPS,
    )
else:
    _train_args.update(
        eval_strategy='epoch',
        save_strategy='epoch',
    )

training_args = Seq2SeqTrainingArguments(**_train_args)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f'Training {RUN_NAME}...')
print(f'  Epochs: {EPOCHS}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Learning rate: {LEARNING_RATE}')
print(f'  Output: {model_output_dir}')
print()

train_result = trainer.train()

print(f'\nTraining complete!')
print(f'  Total steps: {train_result.global_step}')
print(f'  Training loss: {train_result.training_loss:.4f}')

Training nllb-kirundi-multi-20260418...
  Epochs: 5
  Batch size: 1
  Learning rate: 5e-05
  Output: ./models/nllb-kirundi-multi-20260418



/Users/samandari/Documents/Personal/Projects/Kirundi_Dataset/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Bleu
1200,26.505659,2.041709,11.020000
2400,19.304982,2.065710,11.240000
2915,18.174357,2.071056,11.190000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/samandari/Documents/Personal/Projects/Kirundi_Dataset/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/samandari/Documents/Personal/Projects/Kirundi_Dataset/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



Training complete!
  Total steps: 2915
  Training loss: 27.1264


## 9. Evaluate on Test Set

**Evalue le modele sur les ~1,168 paires de test — jamais vues pendant le training.**

C'est le score BLEU **reel** du modele. Le score sur le train/validation peut etre optimiste
(le modele a vu ces donnees). Le score test est la vraie mesure de performance.

Le BLEU est calcule sur toutes les directions melangees (KR→FR, FR→KR, KR→EN, EN→KR).

Tu devrais voir :
- **Test BLEU** : le score de qualite (objectif > 15 pour un premier modele)
- **Test Loss** : l'erreur (plus bas = mieux)

In [16]:
import transformers
print("transformers", transformers.__version__)

test_results = trainer.evaluate(tokenized['test'])
print(f'Test BLEU: {test_results.get("eval_bleu", "N/A")}')
print(f'Test Loss: {test_results["eval_loss"]:.4f}')

transformers 5.5.0
Test BLEU: 9.94
Test Loss: 2.1276


## 10. Save Final Model

**Sauvegarde le meilleur modele et son tokenizer sur le disque Colab.**

Le modele est sauvegarde dans `./models/nllb-kirundi-multi-YYYYMMDD-final/`.

> **Attention sur Colab** : les fichiers sont perdus quand la session se termine.
> C'est pour ca qu'on push vers HuggingFace Hub a l'etape 13.

In [17]:
final_dir = os.path.join(OUTPUT_DIR, f'{RUN_NAME}-final')
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

print(f'Model saved to: {final_dir}')
print(f'To reload later:')
print(f'  tokenizer = AutoTokenizer.from_pretrained("{final_dir}")')
print(f'  model = AutoModelForSeq2SeqLM.from_pretrained("{final_dir}")')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./models/nllb-kirundi-multi-20260418-final
To reload later:
  tokenizer = AutoTokenizer.from_pretrained("./models/nllb-kirundi-multi-20260418-final")
  model = AutoModelForSeq2SeqLM.from_pretrained("./models/nllb-kirundi-multi-20260418-final")


## 11. Interactive Translation

**Teste ton modele dans les 4 directions !**

La fonction `translate(text, src_lang, tgt_lang)` prend n'importe quelle phrase
et la traduit dans la direction voulue. On teste chaque direction avec des exemples.

> Si les resultats sont mauvais, c'est normal avec ~2,900 phrases de base.
> La qualite s'ameliorera significativement avec plus de donnees (objectif : 10,000+ phrases).

In [18]:
def translate(text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors='pt', max_length=MAX_LENGTH, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    tgt_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    output = model.generate(
        **inputs,
        forced_bos_token_id=tgt_token_id,
        max_length=MAX_LENGTH,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


test_cases = [
    ('run_Latn', 'fra_Latn', 'KR->FR', [
        "Amahoro y'Imana abane nawe.",
        "Uburundi ni igihugu ciza.",
        "Ndagukunda.",
    ]),
    ('fra_Latn', 'run_Latn', 'FR->KR', [
        "Comment vas-tu ?",
        "Le Burundi est un beau pays.",
        "Je t'aime.",
    ]),
    ('run_Latn', 'eng_Latn', 'KR->EN', [
        "Ndashaka kujana nawe.",
        "Ijoro ritanga impanuro.",
        "Imana iragukunda.",
    ]),
    ('eng_Latn', 'run_Latn', 'EN->KR', [
        "Peace be with you.",
        "Burundi is a beautiful country.",
        "I love you.",
    ]),
]

for src_lang, tgt_lang, label, sentences in test_cases:
    print(f'=== {label} ({src_lang} -> {tgt_lang}) ===\n')
    for sentence in sentences:
        result = translate(sentence, src_lang, tgt_lang)
        print(f'  IN:  {sentence}')
        print(f'  OUT: {result}')
        print()
    print('-' * 60)

=== KR->FR (run_Latn -> fra_Latn) ===

  IN:  Amahoro y'Imana abane nawe.
  OUT: Que la paix de Dieu soit avec vous!

  IN:  Uburundi ni igihugu ciza.
  OUT: Le Burundi est un beau pays.

  IN:  Ndagukunda.
  OUT: Je t'aime.

------------------------------------------------------------
=== FR->KR (fra_Latn -> run_Latn) ===

  IN:  Comment vas-tu ?
  OUT: Mwariko murakora gute?

  IN:  Le Burundi est un beau pays.
  OUT: Uburundi ni igihugu ciza.

  IN:  Je t'aime.
  OUT: Ndakukunze.

------------------------------------------------------------
=== KR->EN (run_Latn -> eng_Latn) ===

  IN:  Ndashaka kujana nawe.
  OUT: I want to go with you.

  IN:  Ijoro ritanga impanuro.
  OUT: The night brings advice.

  IN:  Imana iragukunda.
  OUT: God loves you.

------------------------------------------------------------
=== EN->KR (eng_Latn -> run_Latn) ===

  IN:  Peace be with you.
  OUT: Amahoro abe muri mwebwe.

  IN:  Burundi is a beautiful country.
  OUT: Uburundi ni igihugu ciza.

  IN:  

## 12. Batch Translate Untranslated Rows

**Utilise le modele entraine pour generer des traductions manquantes dans `metadata.csv`.**

On cherche les lignes qui ont du texte Kirundi mais pas de traduction Francaise et/ou Anglaise,
et on genere des suggestions AI pour chacune.

Deux fichiers CSV sont produits :
- `ai_translations_kr-fra_YYYYMMDD.csv` — traductions Kirundi → Francais
- `ai_translations_kr-eng_YYYYMMDD.csv` — traductions Kirundi → Anglais

> **IMPORTANT** : Ces traductions sont des **suggestions AI** — un humain **doit** les verifier
> avant de les merger dans le dataset.

In [19]:
df_full = pd.read_csv(FILE_PATH)
filled = lambda s: s.notna() & (s.astype(str).str.strip() != '')
datestamp = datetime.now().strftime('%Y%m%d')

batch_targets = [
    ('French_Translation', 'fra_Latn', 'kr-fra'),
    ('English_Translation', 'eng_Latn', 'kr-eng'),
]

for tgt_col, tgt_lang, direction_label in batch_targets:
    needs = df_full[
        filled(df_full['Kirundi_Transcription'])
        & ~filled(df_full[tgt_col])
    ].copy()

    print(f'\n=== {direction_label.upper()} ===')
    print(f'Rows needing {tgt_col}: {len(needs)}')

    if len(needs) == 0:
        print('Nothing to translate!')
        continue

    translations = []
    for i, (_, row) in enumerate(needs.iterrows()):
        result = translate(row['Kirundi_Transcription'], 'run_Latn', tgt_lang)
        translations.append(result)
        if (i + 1) % 100 == 0:
            print(f'  Translated {i + 1}/{len(needs)}...')

    needs['AI_Translation'] = translations

    output_file = f'ai_translations_{direction_label}_{datestamp}.csv'
    needs[['Kirundi_Transcription', tgt_col, 'AI_Translation', 'Domain']].to_csv(
        output_file, index=False, encoding='utf-8-sig',
    )

    print(f'Saved {len(translations)} translations to: {output_file}')
    print('WARNING: Un humain doit verifier avant de merger dans metadata.csv')
    print()
    for _, row in needs.head(3).iterrows():
        print(f'  KR: {row["Kirundi_Transcription"][:70]}')
        print(f'  AI: {row["AI_Translation"][:70]}')
        print()


=== KR-FRA ===
Rows needing French_Translation: 1826
  Translated 100/1826...
  Translated 200/1826...


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/samandari/Documents/Personal/Projects/Kirundi_Dataset/.venv/lib/python3.14/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/samandari/Documents/Personal/Projects/Kirundi_Dataset/.venv/lib/python3.14/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/Users/samandari/Documents/Personal/Projects/Kirundi_Dataset/.venv/lib/python3.14/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/samandari/Documents/Personal/Projects/Kirundi_Dataset/.venv/lib/python3.14/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:torna

  Translated 300/1826...
  Translated 400/1826...
  Translated 500/1826...
  Translated 600/1826...
  Translated 700/1826...
  Translated 800/1826...
  Translated 900/1826...
  Translated 1000/1826...
  Translated 1100/1826...
  Translated 1200/1826...
  Translated 1300/1826...
  Translated 1400/1826...
  Translated 1500/1826...
  Translated 1600/1826...
  Translated 1700/1826...
  Translated 1800/1826...
Saved 1826 translations to: ai_translations_kr-fra_20260419.csv

  KR: Aba na bo bakaba bari mu karere Uburundi busanzwe burimwo, mu bihugu b
  AI: Ceux-ci se trouvent également dans la région où le Burundi se trouve h

  KR: Abacuzi batanyiyegereje ntibacura kuko mbanza kuborohereza ivyuma.
  AI: Les passagers qui ne viennent pas près de moi ne peuvent pas payer par

  KR: Abafise amanota meza mu ndimi n'ibindi vyirwa bijanye na bo bashobora 
  AI: Ceux qui ont de bons résultats en langues et autres domaines connexes 


=== KR-ENG ===
Rows needing English_Translation: 1826
  Translat

## 13. Push to Hugging Face Hub

**Publie le modele multilingue sur le compte HuggingFace `Ijwi-ry-Ikirundi-AI`.**

La cellule suivante :
1. Te demande de te connecter avec ton token HuggingFace (fenetre interactive)
2. Push le modele + tokenizer vers `https://huggingface.co/Ijwi-ry-Ikirundi-AI/nllb-kirundi-multi`
3. Apres le push, n'importe qui peut utiliser le modele :

```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Ijwi-ry-Ikirundi-AI/nllb-kirundi-multi")
model = AutoModelForSeq2SeqLM.from_pretrained("Ijwi-ry-Ikirundi-AI/nllb-kirundi-multi")

tokenizer.src_lang = "run_Latn"
inputs = tokenizer("Amahoro y'Imana abane nawe.", return_tensors="pt")
tgt_id = tokenizer.convert_tokens_to_ids("fra_Latn")
output = model.generate(**inputs, forced_bos_token_id=tgt_id)
print(tokenizer.decode(output[0], skip_special_tokens=True))
```

> **Token requis** : utilise le token **Kirundi-AI-Write-Token** avec permissions d'ecriture.
> Cree-le sur https://huggingface.co/settings/tokens si besoin.

In [1]:
import os, getpass
from huggingface_hub import HfApi, login

token = os.environ.get("HF_TOKEN") or getpass.getpass("HF Token: ")
assert token.startswith("hf_"), "Need a Hugging Face access token"
login(token=token, add_to_git_credential=False)
print("whoami:", HfApi(token=token).whoami())

whoami: {'type': 'user', 'id': '689ca425740b3276357fd0b1', 'name': 'samandari', 'fullname': 'Sama-ndari', 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/689ca425740b3276357fd0b1/95-O-kYqv3wUKxn-nbvmJ.jpeg', 'orgs': [{'type': 'org', 'id': '691c2d91687ad5d86d7372cf', 'name': 'Ijwi-ry-Ikirundi-AI', 'fullname': "Ijwi ry'Ikirundi AI", 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/689ca425740b3276357fd0b1/bVRX2fukra_uVVGiuko47.jpeg'}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Kirundi-AI', 'role': 'fineGrained', 'createdAt': '2026-04-18T23:55:14.894Z', 'fineGrained': {'canReadGatedRepos': False, 'global': [], 'scoped': [{'entity': {'_id': '6912e7ad1a3efda3432035f7', 'type': 'dataset', 'name': 'Ijwi-ry-Ikirundi-AI/Kirundi_Open_Speech_Dataset'}, 'permissions': ['repo.content.read', 'discussion.write', 'repo.write']}, {'entity': {'_id': '689ca425740b3276357fd0b1', 'type': 'user', 'name': 'samandari'}, 'permi

In [ ]:
import os
from huggingface_hub import login


HF_MODEL_REPO = "Ijwi-ry-Ikirundi-AI/nllb-kirundi-multi"
_token = token  
login(token=_token, add_to_git_credential=False)

trainer.args.hub_model_id = HF_MODEL_REPO
trainer.hub_model_id = None
trainer.init_hf_repo(token=_token)

trainer.push_to_hub(
    commit_message="Fine-tuned NLLB-200 for Kirundi multi-direction translation",
    token=_token,
)

tokenizer.push_to_hub(
    HF_MODEL_REPO,
    commit_message="Sync tokenizer with fine-tuned NLLB Kirundi multi model",
    token=_token,
)

print(f"Modele publie sur: https://huggingface.co/{HF_MODEL_REPO}")

In [ ]:
# NO_RETRAIN — upload local folder to Hugging Face Hub
import os
from pathlib import Path
from huggingface_hub import HfApi, login, upload_folder

HF_MODEL_REPO = "Ijwi-ry-Ikirundi-AI/nllb-kirundi-multi"
_token = os.environ.get("HF_TOKEN") or getpass.getpass("HF Token: ")
assert _token and _token.startswith("hf_"), "Set HF_TOKEN to a write-capable token"

# Prefer the -final bundle if you already ran §10 (cell starting with: final_dir = os.path.join(OUTPUT_DIR, f'{RUN_NAME}-final')).
# Otherwise use the BEST checkpoint folder (you had best BLEU ~step 2400 → often checkpoint-2400).
RUN = "nllb-kirundi-multi-20260418"  # change if your folder name differs
candidates = [
    Path("./models") / f"{RUN}-final",
    Path("./models") / RUN / "checkpoint-2400",
    Path("./models") / RUN / "checkpoint-1200",
]
local_dir = next((p for p in candidates if p.is_dir() and (p / "config.json").is_file()), None)
if local_dir is None:
    base = Path("./models") / RUN
    cps = sorted(base.glob("checkpoint-*"))
    raise SystemExit(f"No config.json in tried paths. Under {base} you have: {[p.name for p in cps]}")

print("Uploading from:", local_dir.resolve())

login(token=_token, add_to_git_credential=False)
api = HfApi(token=_token)
api.create_repo(HF_MODEL_REPO, exist_ok=True, repo_type="model")

# Version corrigée sans 'multi_commits'
upload_folder(
    repo_id=HF_MODEL_REPO,
    folder_path=str(local_dir),
    token=_token,
    repo_type="model",
    commit_message="Fine-tuned NLLB Kirundi multi (v1 - final weights)",
    # On retire multi_commits et run_as_future qui peuvent poser problème
)

print("Done. Hub:", f"https://huggingface.co/{HF_MODEL_REPO}")

Uploading from: /Users/samandari/Documents/Personal/Projects/Kirundi_Dataset/scripts/models/nllb-kirundi-multi-20260418-final


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [ ]:
# NO_RETRAIN — Upload résilient fichier par fichier
import os
from pathlib import Path
from huggingface_hub import HfApi, login

HF_MODEL_REPO = "Ijwi-ry-Ikirundi-AI/nllb-kirundi-multi"
_token = os.environ.get("HF_TOKEN") or getpass.getpass("HF Token: ")
assert _token and _token.startswith("hf_"), "Set HF_TOKEN to a write-capable token"

RUN = "nllb-kirundi-multi-20260418"
candidates = [
    Path("./models") / f"{RUN}-final",
    Path("./models") / RUN / "checkpoint-2400",
]

local_dir = next((p for p in candidates if p.is_dir() and (p / "config.json").is_file()), None)
if local_dir is None:
    raise SystemExit("Dossier local introuvable ou config.json manquant.")

print(f"🚀 Préparation de l'upload depuis : {local_dir.resolve()}")

login(token=_token, add_to_git_credential=False)
api = HfApi(token=_token)
api.create_repo(HF_MODEL_REPO, exist_ok=True, repo_type="model")

# Liste tous les fichiers dans le dossier
files_to_upload = [f for f in local_dir.rglob("*") if f.is_file()]

print(f"📦 {len(files_to_upload)} fichiers à envoyer...")

for file_path in files_to_upload:
    relative_path = file_path.relative_to(local_dir)
    print(f"⏳ Uploading: {relative_path}...")
    
    try:
        api.upload_file(
            path_or_fileobj=str(file_path),
            path_in_repo=str(relative_path),
            repo_id=HF_MODEL_REPO,
            token=_token,
            commit_message=f"Upload {relative_path} (resilient mode)"
        )
        print(f"✅ {relative_path} terminé.")
    except Exception as e:
        print(f"❌ Erreur sur {relative_path}: {e}")
        print("💡 Relance la cellule pour reprendre là où ça a coupé.")

print(f"\n✨ Terminé ! Hub: https://huggingface.co/{HF_MODEL_REPO}")